In [ ]:
import sys; sys.path.append('..')
import MeshFEM
import mesh, elastic_sheet, energy, benchmark
import triangulation
from tri_mesh_viewer import TriMeshViewer
import numpy as np
import mesh_operations

In [ ]:
# Generate a mesh of the unit disk
radius = 10 # 10mm
thetas = np.linspace(0, 2 * np.pi, 100, endpoint=False)
m = mesh.Mesh(*triangulation.triangulate(
    radius * np.array([np.cos(thetas), np.sin(thetas)]).T,
    np.array([np.arange(len(thetas)), np.roll(np.arange(len(thetas)), -1)]).T, outputPointMarkers=False, triArea=1))

# Glue mesh to itself along the boundary by creating a merged mesh where all internal vertices have been perturbed imperceptibly
# out-of-plane to prevent their merging.
isInternal = np.ones(m.numVertices(), dtype=bool)
isInternal[m.boundaryVertices()] = False

Vtop = m.vertices().copy()
Vbot = m.vertices().copy()
Vtop[isInternal, 2] =  1e-16 # np.sqrt(1 - np.linalg.norm(Vtop[isInternal, :] / radius, axis=1)**2)
Vbot[isInternal, 2] = -1e-16 # -np.sqrt(1 - np.linalg.norm(Vtop[isInternal, :] / radius, axis=1)**2)

Ftop = m.elements().copy()
Fbot = m.elements().copy()
Fbot[:, [0, 1]] = Fbot[:, [1, 0]]

m = mesh.Mesh(*mesh_operations.mergedMesh([(Vtop, Ftop), (Vbot, Fbot)]))

In [ ]:
psi = energy.NeoHookeanYoungPoisson(2, 200, 0.3) # Y = 200MPa, nu = 0.3
es = elastic_sheet.ElasticSheet(m, psi)
es.thickness = 0.1 # 0.1mm
pinVars, pinVerts = es.prepareRigidMotionPins()

In [ ]:
esview = TriMeshViewer(es, wireframe=True, width=1024, height=768)
esview.materialLibrary.material(False).color='#3E0'
esview.show()

In [ ]:
# Configure visualization
visNormals = False
nview = None
def updateNormalView():
    from tri_mesh_viewer import PointCloudViewer
    global nview
    if not visNormals: return
    esview.subViews = []
    nview = PointCloudViewer(es.edgeMidpoints(), vectorField=es.midedgeNormals(), superView=esview)
    nview.arrowSize = 30
updateNormalView()

# Callback to update visualization during equilibrium solve
def iter_cb(prob, it):
    if (it % 5 == 1):
       esview.update()
       updateNormalView()

In [ ]:
import loads
inflation = loads.Inflation(es)

In [ ]:
inflation.pressure = 0.01
es.computeEquilibrium(loads=[inflation], fixedVars=pinVars, cb=iter_cb)

## Finite Difference Validation

In [ ]:
import fd_validation
prob = es.EquilibriumProblem([inflation])

In [ ]:
fd_validation.gradConvergencePlot(prob)

In [ ]:
prob.invalidateCachedHessian()
fd_validation.hessConvergencePlot(prob)